# Linear and Quadratic Discriminant Analysis

## Overview

**Discriminant analysis** is a family of generative classification methods that model the class-conditional distribution of features. The decision rule is derived from Bayes' theorem, yielding interpretable decision boundaries and direct access to posterior probabilities.

### Gaussian generative model

We assume that feature vectors $x \in \mathbb{R}^d$ within class $k$ follow a multivariate Gaussian distribution:
$$p(x \mid y = k) = \mathcal{N}(x; \mu_k, \Sigma_k) = \frac{1}{(2\pi)^{d/2}|\Sigma_k|^{1/2}} \exp\!\left(-\frac{1}{2}(x - \mu_k)^T \Sigma_k^{-1} (x - \mu_k)\right)$$

with prior $\pi_k = P(y = k)$. By Bayes' rule, the posterior is:
$$P(y = k \mid x) \propto p(x \mid y = k) \, \pi_k$$

### Linear Discriminant Analysis (LDA)

LDA assumes **all classes share the same covariance matrix** $\Sigma_k = \Sigma$. The log-posterior of class $k$ reduces to a linear discriminant function:
$$\delta_k(x) = x^T \Sigma^{-1} \mu_k - \frac{1}{2}\mu_k^T \Sigma^{-1} \mu_k + \log \pi_k$$

The decision boundary between classes $j$ and $k$ is **linear** in $x$: $\delta_j(x) = \delta_k(x)$.

### Quadratic Discriminant Analysis (QDA)

QDA allows **class-specific covariances** $\Sigma_k$. The log-posterior is:
$$\delta_k(x) = -\frac{1}{2}\log|\Sigma_k| - \frac{1}{2}(x - \mu_k)^T \Sigma_k^{-1}(x - \mu_k) + \log \pi_k$$

The decision boundary between classes $j$ and $k$ is now **quadratic** in $x$.

### Fisher's linear projection

LDA also provides a projection direction $w$ maximizing class separability relative to within-class scatter:
$$w^* = \arg\max_w \frac{w^T S_B w}{w^T S_W w}$$

where $S_B = \sum_k n_k (\mu_k - \mu)(\mu_k - \mu)^T$ is the **between-class scatter matrix** and $S_W = \sum_k \sum_{i:y_i=k}(x_i - \mu_k)(x_i - \mu_k)^T$ is the **within-class scatter matrix**. The optimal directions are eigenvectors of $S_W^{-1} S_B$.

### What this notebook demonstrates

1. Fitting LDA and QDA to 2D data with 3 classes.
2. Decision region visualization and comparison.
3. Fisher's projection onto the discriminant axis.
4. Parameter study: effect of class separation and covariance structure.

### Imports

We implement LDA and QDA from scratch using `numpy` for clarity, and also use `sklearn.discriminant_analysis` for validation.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

rng = np.random.default_rng(0)

### Dataset generation

We generate a 2D, 3-class dataset from Gaussian distributions. The three class means are placed at the vertices of a triangle in $\mathbb{R}^2$. For the initial configuration:
- Class 0: spherical covariance.
- Class 1: elongated (larger variance along $y$).
- Class 2: rotated ellipse.

This ensures the QDA decision boundary is distinctly non-linear while LDA gives a coarser but interpretable linear boundary.

In [2]:
def make_3class_data(means, covs, n_per_class=200, seed=0):
    rng = np.random.default_rng(seed)
    X_list, y_list = [], []
    for k, (mu, Sigma) in enumerate(zip(means, covs)):
        Xk = rng.multivariate_normal(mu, Sigma, size=n_per_class)
        X_list.append(Xk)
        y_list.append(np.full(n_per_class, k))
    return np.vstack(X_list), np.concatenate(y_list)

# Class parameters
means0 = [np.array([-2.0, 0.0]),
           np.array([2.0, 1.5]),
           np.array([0.5, -2.5])]

R = lambda t: np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
covs0 = [
    0.5 * np.eye(2),
    R(np.pi / 4) @ np.diag([1.5, 0.3]) @ R(np.pi / 4).T,
    np.diag([0.4, 1.2]),
]

X, y = make_3class_data(means0, covs0)

colors = ['tab:blue', 'tab:orange', 'tab:green']
fig, ax = plt.subplots(figsize=(6, 5))
for k in range(3):
    mask = y == k
    ax.scatter(X[mask, 0], X[mask, 1], s=15, alpha=0.5,
               color=colors[k], label=f'Class {k}')
ax.set_title('Training data (3 classes)')
ax.legend()
plt.tight_layout()
plt.savefig('data.png', dpi=80, bbox_inches='tight')
plt.close()

### Decision region visualization

We plot decision regions for both LDA and QDA. For any point $x$ on a dense grid, the predicted class is $\hat{k}(x) = \arg\max_k \delta_k(x)$.

**LDA** partitions the plane into convex regions separated by linear boundaries (hyperplanes).
**QDA** can produce curved (conic section) boundaries, allowing non-convex class regions that better match the true class distributions.

In [3]:
def plot_decision_regions(ax, clf, X, y, title, colors, resolution=300):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                          np.linspace(y_min, y_max, resolution))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = clf.predict(grid).reshape(xx.shape)
    cmap_bg = ListedColormap([c + '55' for c in ['#4e79a7', '#f28e2b', '#59a14f']])
    ax.contourf(xx, yy, Z, alpha=0.35, cmap=cmap_bg)
    ax.contour(xx, yy, Z, levels=[0.5, 1.5], colors='k', linewidths=1.0)
    for k in range(3):
        mask = y == k
        ax.scatter(X[mask, 0], X[mask, 1], s=12, alpha=0.6,
                   color=colors[k], zorder=3)
    ax.set_title(title, fontsize=11)
    ax.set_aspect('equal')

lda = LinearDiscriminantAnalysis()
qda = QuadraticDiscriminantAnalysis()
lda.fit(X, y)
qda.fit(X, y)

print(f'LDA accuracy: {lda.score(X, y):.3f}')
print(f'QDA accuracy: {qda.score(X, y):.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_decision_regions(axes[0], lda, X, y, 'LDA (linear boundary)', colors)
plot_decision_regions(axes[1], qda, X, y, 'QDA (quadratic boundary)', colors)
plt.tight_layout()
plt.savefig('decision_regions.png', dpi=80, bbox_inches='tight')
plt.close()

LDA accuracy: 0.978
QDA accuracy: 0.978


### Fisher's linear discriminant projection

The Fisher projection maximizes between-class variance relative to within-class variance. For $K$ classes, there are at most $\min(K-1, d)$ discriminant directions.

We compute the projection $W^* = S_W^{-1/2} \text{eig}(S_W^{-1/2} S_B S_W^{-1/2})$ directly and project the data onto the first 2 Fisher directions, visualizing the projected distribution as a 1D histogram.

The scatter matrices are:
$$S_W = \sum_{k=1}^K \sum_{i : y_i = k} (x_i - \mu_k)(x_i - \mu_k)^T$$
$$S_B = \sum_{k=1}^K n_k (\mu_k - \bar{\mu})(\mu_k - \bar{\mu})^T$$

In [4]:
def fisher_lda(X, y):
    """Compute Fisher LDA projection directions."""
    classes = np.unique(y)
    n, d = X.shape
    mu_global = X.mean(axis=0)
    S_W = np.zeros((d, d))
    S_B = np.zeros((d, d))
    mus = {}
    for k in classes:
        Xk = X[y == k]
        mu_k = Xk.mean(axis=0)
        mus[k] = mu_k
        diff = Xk - mu_k
        S_W += diff.T @ diff
        n_k = len(Xk)
        d_mu = (mu_k - mu_global)[:, None]
        S_B += n_k * (d_mu @ d_mu.T)
    # Generalized eigenproblem: S_B w = lambda S_W w
    S_W_inv = np.linalg.inv(S_W + 1e-8 * np.eye(d))
    M = S_W_inv @ S_B
    evals, evecs = np.linalg.eig(M)
    idx = np.argsort(-np.real(evals))
    return np.real(evals[idx]), np.real(evecs[:, idx]), mus

evals_f, evecs_f, mus_f = fisher_lda(X, y)
print(f'Fisher eigenvalues: {evals_f[:2].round(3)}')

# Project onto first two Fisher directions
W_fisher = evecs_f[:, :2]
X_proj = X @ W_fisher

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter in original space with Fisher directions
for k in range(3):
    mask = y == k
    axes[0].scatter(X[mask, 0], X[mask, 1], s=12, alpha=0.5, color=colors[k])
origin = X.mean(axis=0)
scale = 3.0
for i, (c, lbl) in enumerate(zip(['purple', 'brown'], ['Fisher 1', 'Fisher 2'])):
    w = W_fisher[:, i]
    axes[0].annotate('', xy=origin + scale * w, xytext=origin,
                     arrowprops=dict(arrowstyle='->', color=c, lw=2.5))
    axes[0].text(*(origin + scale * w * 1.1), lbl, color=c, fontsize=10)
axes[0].set_title('Fisher discriminant directions')
axes[0].set_aspect('equal')

# Projected distribution
for k in range(3):
    mask = y == k
    axes[1].hist(X_proj[mask, 0], bins=30, alpha=0.5, color=colors[k],
                 label=f'Class {k}', density=True)
axes[1].set_xlabel('Fisher direction 1')
axes[1].set_ylabel('Density')
axes[1].set_title('Projected distribution (Fisher direction 1)')
axes[1].legend()

plt.tight_layout()
plt.savefig('fisher_projection.png', dpi=80, bbox_inches='tight')
plt.close()

Fisher eigenvalues: [5.265 3.438]


### Parameter study: effect of class separation

We vary the inter-class distance $\Delta$ (scaling the mean separation) and compute both LDA and QDA train accuracy. As $\Delta$ increases, the classes become linearly separable and LDA approaches perfect accuracy. QDA benefits from class-specific covariances even at moderate separation.

We also show the decision boundaries at three representative separations side by side.

In [5]:
deltas = np.linspace(0.5, 4.0, 12)
lda_acc = []
qda_acc = []

base_means = [np.array([-1.0, 0.0]), np.array([1.0, 0.8]), np.array([0.25, -1.3])]

for delta in deltas:
    scaled_means = [delta * m for m in base_means]
    Xi, yi = make_3class_data(scaled_means, covs0, n_per_class=150)
    lda_i = LinearDiscriminantAnalysis().fit(Xi, yi)
    qda_i = QuadraticDiscriminantAnalysis().fit(Xi, yi)
    lda_acc.append(lda_i.score(Xi, yi))
    qda_acc.append(qda_i.score(Xi, yi))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(deltas, lda_acc, 'o-', label='LDA', color='tab:blue')
axes[0].plot(deltas, qda_acc, 's-', label='QDA', color='tab:orange')
axes[0].set_xlabel('Class separation $\\Delta$')
axes[0].set_ylabel('Train accuracy')
axes[0].set_title('Accuracy vs. class separation')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Decision regions at two extreme separations
for axi, delta in zip([axes[1]], [2.0]):
    scaled_means = [delta * m for m in base_means]
    Xi, yi = make_3class_data(scaled_means, covs0, n_per_class=150)
    lda_i = LinearDiscriminantAnalysis().fit(Xi, yi)
    qda_i = QuadraticDiscriminantAnalysis().fit(Xi, yi)
    plot_decision_regions(axi, qda_i, Xi, yi, f'QDA, $\\Delta={delta}$', colors)

plt.tight_layout()
plt.savefig('param_study.png', dpi=80, bbox_inches='tight')
plt.close()

### Effect of covariance structure on decision boundaries

We compare how LDA and QDA handle three covariance scenarios:
1. **Spherical**: all classes have $\Sigma_k = \sigma^2 I$ (LDA and QDA agree).
2. **Shared anisotropic**: all classes have the same elongated $\Sigma$ (LDA optimal).
3. **Class-specific anisotropic**: each class has a different $\Sigma_k$ (QDA necessary).

This illustrates the **bias-variance tradeoff** inherent in the LDA/QDA choice: LDA has fewer parameters (one shared $\Sigma$) and is preferable when $n \ll d$ or when the homoscedasticity assumption holds.

In [6]:
R30 = R(np.pi / 6)
shared_cov = R30 @ np.diag([1.0, 0.25]) @ R30.T

scenarios = [
    ('Spherical', [0.4 * np.eye(2)] * 3),
    ('Shared anisotropic', [shared_cov] * 3),
    ('Class-specific', covs0),
]

fixed_means = [np.array([-2.0, 0.0]), np.array([2.0, 1.5]), np.array([0.5, -2.5])]

fig, axes = plt.subplots(3, 2, figsize=(12, 15))
for row, (scenario_name, covs_sc) in enumerate(scenarios):
    Xsc, ysc = make_3class_data(fixed_means, covs_sc, n_per_class=200)
    lda_sc = LinearDiscriminantAnalysis().fit(Xsc, ysc)
    qda_sc = QuadraticDiscriminantAnalysis().fit(Xsc, ysc)
    plot_decision_regions(axes[row, 0], lda_sc, Xsc, ysc,
                          f'{scenario_name}: LDA (acc={lda_sc.score(Xsc,ysc):.2f})', colors)
    plot_decision_regions(axes[row, 1], qda_sc, Xsc, ysc,
                          f'{scenario_name}: QDA (acc={qda_sc.score(Xsc,ysc):.2f})', colors)

plt.suptitle('LDA vs QDA: effect of covariance structure', fontsize=12)
plt.tight_layout()
plt.savefig('covariance_comparison.png', dpi=70, bbox_inches='tight')
plt.close()

### Interactive class configuration

Use the sliders to adjust the separation of class means and the noise level $\sigma$. The decision boundaries for LDA and QDA are updated in real time.

### Static snapshot

In [7]:
STATIC_SNAPSHOT = True
if STATIC_SNAPSHOT:
    fig, axes = plt.subplots(3, 2, figsize=(12, 15))
    for row, (scenario_name, covs_sc) in enumerate(scenarios):
        Xsc, ysc = make_3class_data(fixed_means, covs_sc, n_per_class=200)
        lda_sc = LinearDiscriminantAnalysis().fit(Xsc, ysc)
        qda_sc = QuadraticDiscriminantAnalysis().fit(Xsc, ysc)
        plot_decision_regions(axes[row, 0], lda_sc, Xsc, ysc,
                              f'{scenario_name}\nLDA acc={lda_sc.score(Xsc,ysc):.2f}', colors)
        plot_decision_regions(axes[row, 1], qda_sc, Xsc, ysc,
                              f'{scenario_name}\nQDA acc={qda_sc.score(Xsc,ysc):.2f}', colors)
    plt.suptitle('LDA vs QDA Decision Regions', fontsize=13)
    plt.tight_layout()
    plt.savefig('snippet.png', dpi=100, bbox_inches='tight')
    plt.close()

## Takeaways

- **LDA** assumes a shared covariance matrix across classes, leading to linear decision boundaries and an efficient closed-form solution.
- **QDA** allows class-specific covariances, enabling curved (conic section) boundaries at the cost of estimating $K$ separate covariance matrices.
- When class covariances are truly equal (homoscedasticity), LDA is statistically more efficient than QDA.
- **Fisher's projection** provides the optimal direction(s) for class separation, with at most $K-1$ discriminant directions.
- Both methods are **generative**: they model $p(x|y=k)$ and use Bayes' theorem, unlike discriminative methods (logistic regression, SVM) which model $P(y|x)$ directly.
- QDA is sensitive to covariance estimation: when $n/d$ is small, regularized QDA (RDA) or shrinkage estimators improve stability.

## Bibliography

- R. A. Fisher, *The use of multiple measurements in taxonomic problems*, Annals of Eugenics, 7(2):179–188, 1936.
- T. Hastie, R. Tibshirani, J. Friedman, *The Elements of Statistical Learning*, Springer, 2nd edition, 2009. Section 4.3.
- C. M. Bishop, *Pattern Recognition and Machine Learning*, Springer, 2006. Chapter 4.
- J. H. Friedman, *Regularized discriminant analysis*, Journal of the American Statistical Association, 84(405):165–175, 1989.